# REDS Dataset Pair Verification, Integrity Validation & EDA

This notebook presents the exploratory data analysis (EDA), 1-to-1 sequence and frame pairing verification, full image-integrity checks, and visual inspections for the REDS video deblurring research project.

## 1. Project Configuration and Path Resolution
Loads `configs/eda_config.yaml` dynamically without machine-specific hardcoded paths.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Resolve project root dynamically
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
from src.data.config_utils import get_project_root, load_config

project_root = get_project_root()
config = load_config(project_root=project_root)
print(f"Project root resolved to: {project_root}")
print(f"Visualization display counts: Train={config['visualization']['train_pair_count']}, Val={config['visualization']['val_pair_count']}, Consecutive={config['visualization']['consecutive_frames_per_sequence']}")

## 2. Dataset Inventory Overview
Inspects all discovered datasets, sequences, and image counts.

In [ ]:
reports_dir = project_root / config['paths']['reports_dir']
df_inv = pd.read_csv(reports_dir / 'dataset_inventory.csv')
print(f"Discovered total sequence folders: {len(df_inv)}")
inv_summary = df_inv.groupby('dataset').agg(sequences=('sequence', 'count'), total_frames=('image_count', 'sum')).reset_index()
display(inv_summary)

## 3. Sequence and Frame Correspondence Checks
Verifies 1-to-1 matching of sequence directories and frame filenames between `train_blur` $\leftrightarrow$ `train_sharp` and `val_blur` $\leftrightarrow$ `val_sharp`.

In [ ]:
df_pairing_summary = pd.read_csv(reports_dir / 'frame_pairing_summary.csv')
print(f"Total sequences analyzed: {len(df_pairing_summary)}")
print(f"Pairing status distribution:")
display(df_pairing_summary['pairing_status'].value_counts())
display(df_pairing_summary.head(10))

## 4. Full Image-Integrity Validation
Examines the decoding status, readability, and structural integrity of all 54,000 images in training and validation splits.

In [ ]:
df_val = pd.read_csv(reports_dir / 'image_validation_results.csv')
print(f"Total images validated: {len(df_val)}")
print(f"Image status distribution:")
display(df_val['status'].value_counts())
print(f"Image modes: {df_val['mode'].unique()}, Resolutions: {df_val[['width', 'height']].drop_duplicates().to_dict(orient='records')}")

## 5. Matched-Pair Dimension and Structural Checks
Confirms that corresponding blurred and sharp image pairs share identical dimensions and color channels.

In [ ]:
df_mismatches = pd.read_csv(reports_dir / 'image_dimension_mismatches.csv')
print(f"Total dimension / readability mismatches found across matched pairs: {len(df_mismatches)}")
if len(df_mismatches) == 0:
    print("PERFECT MATCH: 100% of candidate pairs share identical dimensions and modes.")
else:
    display(df_mismatches)

## 6. Pairing Issues Summary
Detailed log of any missing sequences, missing frames, or corrupted files.

In [ ]:
df_issues = pd.read_csv(reports_dir / 'frame_pairing_issues.csv')
print(f"Total pairing discrepancies recorded: {len(df_issues)}")
if len(df_issues) == 0:
    print("Zero pairing discrepancies or missing frames found.")
else:
    display(df_issues)

## 7. Configurable Training-Pair Visualization
Displays side-by-side blurred inputs and sharp ground-truth targets from the training split.

In [ ]:
fig_dir = project_root / config['paths']['figures_dir']
train_fig = fig_dir / 'train_blur_sharp_pairs.png'
if train_fig.exists():
    img = Image.open(train_fig)
    plt.figure(figsize=(14, 18))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print("Training pairs figure not found.")

## 8. Configurable Validation-Pair Visualization
Displays side-by-side blurred inputs and sharp ground-truth targets from the validation split.

In [ ]:
val_fig = fig_dir / 'val_blur_sharp_pairs.png'
if val_fig.exists():
    img = Image.open(val_fig)
    plt.figure(figsize=(14, 18))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print("Validation pairs figure not found.")

## 9. Consecutive-Frame Temporal Inspection
Inspects consecutive frames from the same sequence to examine temporal continuity and check for unexpected jumps or misalignments.

In [ ]:
consec_fig = fig_dir / 'consecutive_frame_comparisons.png'
if consec_fig.exists():
    img = Image.open(consec_fig)
    plt.figure(figsize=(14, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print("Consecutive frames figure not found.")

## 10. Visualization Metadata and Selected Image List
Detailed metadata table for all displayed pairs.

In [ ]:
df_vis = pd.read_csv(reports_dir / 'visualized_pairs.csv')
print(f"Total visualized pairs: {len(df_vis)}")
display(df_vis)

## 11. Summary and Dataset Readiness Assessment

### Verification Summary
- **Training:** 240 sequences × 100 frames = 24,000 blur/sharp matched pairs (100% matched).
- **Validation:** 30 sequences × 100 frames = 3,000 blur/sharp matched pairs (100% matched).
- **Integrity:** 54,000 total images validated without a single decoding error, zero-byte file, or corruption.
- **Dimensions:** 100% of images are uniform 1280x720 RGB PNG format.
- **Classification:** `PASS` — The REDS dataset is structurally verified and ready for dataset loaders, sliding window batching, and lightweight deblurring model development.